In [163]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# 🔍 AI FactLens

## AI-Powered Fact Checker using Gemini + RAG + FAISS

---

### 📌 Project Overview

AI FactLens is an intelligent AI assistant that retrieves relevant information, builds a temporary vector database using FAISS, and generates trustworthy answers using Google Gemini.

---

## 🚀 Features

- 🔎 Retrieval-Augmented Generation (RAG)
- 🤖 Google Gemini API
- 🤗 Hugging Face Embeddings
- 📚 FAISS Vector Database
- 📄 Output Parsing
- 🎨 Interactive Gradio GUI
- 🌐 Live Information Retrieval
- 📂 GitHub Ready Notebook

---

# 📦 Install Required Libraries

In [164]:
!pip -q install -U \
google-genai \
sentence-transformers \
langchain-text-splitters \
faiss-cpu \
gradio \
huggingface_hub

# 📚 Import Libraries

In [165]:
import os
import json
import requests
import warnings

import numpy as np
import faiss
import gradio as gr

from google import genai

from sentence_transformers import SentenceTransformer

from langchain_text_splitters import RecursiveCharacterTextSplitter

from huggingface_hub import login

from kaggle_secrets import UserSecretsClient

warnings.filterwarnings("ignore")

print("✅ Libraries Imported Successfully")

✅ Libraries Imported Successfully


# 🔑 Load API Keys

In [166]:
user_secrets = UserSecretsClient()

GEMINI_API_KEY = user_secrets.get_secret("GEMINI_API_KEY")

HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

SERPER_API_KEY = user_secrets.get_secret("SERPER_API_KEY")

print("✅ API Keys Loaded Successfully")

✅ API Keys Loaded Successfully


In [167]:
login(HF_TOKEN)

print("✅ Hugging Face Login Successful")

✅ Hugging Face Login Successful


In [168]:
client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("✅ Gemini Client Ready")

✅ Gemini Client Ready


# 🧠 Load Embedding Model

In [170]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding Model Loaded")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding Model Loaded


# ✂️ Initialize Text Splitter

In [171]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

print("✅ Text Splitter Ready")

✅ Text Splitter Ready


# ✅ Environment Check

In [172]:
print("=" * 60)
print("✅ Gemini Ready")
print("✅ Hugging Face Ready")
print("✅ Embedding Model Ready")
print("✅ Text Splitter Ready")
print("=" * 60)

✅ Gemini Ready
✅ Hugging Face Ready
✅ Embedding Model Ready
✅ Text Splitter Ready


# 🌍 Live Web Search with Serper API

Search the web to retrieve reliable information before generating the answer.

In [173]:
TRUSTED_DOMAINS = [
    "wikipedia.org",
    "who.int",
    "nih.gov",
    "cdc.gov",
    "nasa.gov",
    "un.org",
    "britannica.com",
    "nature.com",
    "science.org"
]

print("✅ Trusted Sources Loaded")

✅ Trusted Sources Loaded


In [174]:
def search_web(query, num_results=5):

    url = "https://google.serper.dev/search"

    payload = json.dumps({
        "q": query,
        "num": num_results
    })

    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    response = requests.post(
        url,
        headers=headers,
        data=payload
    )

    results = response.json()

    trusted_results = []

    for item in results.get("organic", []):

        link = item.get("link", "")

        if any(domain in link for domain in TRUSTED_DOMAINS):

            trusted_results.append({
                "title": item.get("title"),
                "link": link,
                "snippet": item.get("snippet", "")
            })

    return trusted_results

In [175]:
results = search_web(
    "Is coffee good for health?"
)

results

[{'title': "Coffee's Impact on Health and Well-Being - PMC",
  'link': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC12348139/',
  'snippet': 'by RC Emadi · 2025 · Cited by 27 — Coffee is often taken to hydrate the body, enhance sports performance, improve mental acuity, and delay or reduce sleep. It may also increase bowel movement.'}]

# Download Web Pages

Retrieve the content of trusted web pages before building the RAG index.

In [176]:
!pip -q install beautifulsoup4

In [177]:
from bs4 import BeautifulSoup

print("✅ BeautifulSoup Ready")

✅ BeautifulSoup Ready


# Extract Website Content

Clean the HTML and keep only readable text.

In [178]:
def extract_page_text(url):

    try:

        response = requests.get(
            url,
            timeout=20,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for tag in soup([
            "script",
            "style",
            "header",
            "footer",
            "nav",
            "aside"
        ]):
            tag.decompose()

        text = soup.get_text(
            separator=" ",
            strip=True
        )

        return text

    except Exception as e:

        return ""

In [179]:
results = search_web(
    "Does coffee cause dehydration?"
)

text = extract_page_text(
    results[0]["link"]
)

print(text[:1000])

No Evidence of Dehydration with Moderate Daily Coffee Intake: A Counterbalanced Cross-Over Study in a Free-Living Population - PMC Skip to main content Official websites use .gov A .gov website belongs to an official
                            government organization in the United States. Secure .gov websites use HTTPS A lock ( Lock Locked padlock icon ) or https:// means you've safely
                                connected to the .gov website. Share sensitive
                                information only on official, secure websites. Search PMC Full-Text Archive Search in PMC Journal List User Guide PERMALINK Copy As a library, NLM provides access to scientific literature. Inclusion in an NLM database does not imply endorsement of, or agreement with,
    the contents by NLM or the National Institutes of Health. Learn more: PMC Disclaimer | PMC Copyright Notice PLoS One . 2014 Jan 9;9(1):e84154. doi: 10.1371/journal.pone.0084154 No Evidence of Dehydration with Moderate Daily Cof

# Build the Knowledge Base (FAISS)

Convert the extracted text into chunks, generate embeddings,
and store them in a FAISS vector database.

This temporary knowledge base is created for every user query,
allowing the chatbot to verify claims using fresh trusted sources.

In [180]:
def build_vector_store(text):

    chunks = text_splitter.split_text(text)

    embeddings = embedding_model.encode(
        chunks,
        convert_to_numpy=True
    )

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatL2(dimension)

    index.add(embeddings)

    return index, chunks

In [182]:
index, chunks = build_vector_store(text)

print(f"Number of Chunks: {len(chunks)}")

Number of Chunks: 114


# Retrieve Relevant Context

Find the most relevant text chunks using semantic similarity search.

In [181]:
def retrieve_context(question, index, chunks, top_k=5):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        question_embedding,
        top_k
    )

    retrieved_chunks = []

    for idx in indices[0]:
        retrieved_chunks.append(chunks[idx])

    return "\n\n".join(retrieved_chunks)

In [183]:
context = retrieve_context(
    "Does coffee cause dehydration?",
    index,
    chunks
)

print(context)

Thompson : Editor Received 2013 Aug 14; Accepted 2013 Nov 12; Collection date 2014. © 2014 Killer et al This is an open-access article distributed under the terms of the Creative Commons Attribution License, which permits unrestricted use, distribution, and reproduction in any medium, provided the original author and source are properly credited. PMC Copyright notice PMCID: PMC3886980  PMID: 24416202 Abstract It is often suggested that coffee causes dehydration and its consumption should be

Abstract It is often suggested that coffee causes dehydration and its consumption should be avoided or significantly reduced to maintain fluid balance. The aim of this study was to directly compare the effects of coffee consumption against water ingestion across a range of validated hydration assessment techniques. In a counterbalanced cross-over design, 50 male coffee drinkers (habitually consuming 3–6 cups per day) participated in two trials, each lasting three consecutive days. In addition

to d

# Advanced Fact Checking Prompt

The model must verify the claim using only retrieved evidence.

The output is returned as structured JSON that will later
be displayed inside the Gradio interface.

In [186]:
import json

FACT_CHECK_PROMPT = """
You are an expert AI Fact Checker.

Your task is to verify the user's claim using the provided evidence context.

Rules:

1. Use the provided context as the primary source of evidence.
2. Do not mark a claim as True unless the context contains clear supporting evidence.
3. Do not invent facts.
4. If evidence is missing or unclear, return "Not Enough Evidence".
5. Viral rumors without reliable evidence should not be classified as True.
6. Confidence must be an integer between 0 and 100.
7. Bullshit Index must be an integer between 0 and 10.
8. Return ONLY valid JSON.
9. Do not wrap JSON inside markdown.

Verdict must be ONLY one of:

- True
- False
- Misleading
- Not Enough Evidence

Context:
{context}

Claim:
{claim}

Return exactly this JSON format:

{{
    "claim": "",
    "verdict": "",
    "confidence": 0,
    "bullshit_index": 0,
    "reasoning": "",
    "supporting_evidence": "",
    "final_summary": ""
}}
"""


def fact_check(claim, context):
    prompt = FACT_CHECK_PROMPT.format(
        context=context,
        claim=claim
    )

    response = client.models.generate_content(
               model="gemini-3.5-flash-lite",
        contents=prompt
    )

    text = response.text.strip()

    # تم تصحيح التنظيف هنا بشكل صحيح تماماً
    if text.startswith("```json"):
        text = text[7:]
    elif text.startswith("```"):
        text = text[3:]

    if text.endswith("```"):
        text = text[:-3]

    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        return {
            "claim": claim,
            "verdict": "Parsing Error",
            "confidence": 0,
            "bullshit_index": 0,
            "reasoning": text,
            "supporting_evidence": "",
            "final_summary": "Gemini returned invalid JSON."
        }


def verify_claim(claim):
    results = search_web(claim)

    if not results:
        return {
            "error": "No trusted sources found."
        }

    all_text = ""

    for item in results[:5]:
        page_text = extract_page_text(
            item["link"]
        )

        if page_text:
            all_text += page_text + "\n\n"

    index, chunks = build_vector_store(
        all_text
    )

    context = retrieve_context(
        claim,
        index,
        chunks,
        top_k=8
    )

    output = fact_check(
        claim,
        context
    )

    output["sources"] = [
        {
            "title": item["title"],
            "url": item["link"]
        }
        for item in results[:5]
    ]

    return output

In [187]:
result = verify_claim(
    "Coffee causes dehydration."
)

print(json.dumps(result, indent=4))

{
    "claim": "Coffee causes dehydration.",
    "verdict": "False",
    "confidence": 100,
    "bullshit_index": 10,
    "reasoning": "The provided context explicitly refutes the claim that coffee causes dehydration. The study titled 'No Evidence of Dehydration with Moderate Daily Coffee Intake' concluded that coffee, when consumed in moderation by caffeine-habituated males, contributes to daily fluid requirement and does not result in dehydration or pose a detrimental effect to fluid balance.",
    "supporting_evidence": "Our data shows no significant differences in the hydrating properties of coffee or water across a wide range... results suggest that coffee did not result in dehydration when provided in a moderate dose of 4 mg/kg BW caffeine in four cups per day.",
    "final_summary": "Scientific evidence from the provided study shows that moderate daily coffee consumption does not cause dehydration and contributes adequately to daily fluid requirements.",
    "sources": [
       

# Gradio Interface

Launch the AI Fact Checker chatbot.

In [194]:
import gradio as gr
import json


# ═══════════════════════════════════════════════════════════════
# 🎨 ENHANCED CSS - Modern, colorful, glassmorphism design
# ═══════════════════════════════════════════════════════════════
custom_css = """
/* ── Global Reset & Base ── */
* { box-sizing: border-box; }

body {
    background: 
        radial-gradient(ellipse at 0% 0%, #0ea5e9 0%, transparent 40%),
        radial-gradient(ellipse at 100% 0%, #8b5cf6 0%, transparent 40%),
        radial-gradient(ellipse at 100% 100%, #ec4899 0%, transparent 40%),
        radial-gradient(ellipse at 0% 100%, #10b981 0%, transparent 40%),
        #0f172a;
    background-attachment: fixed;
    font-family: 'Inter', 'Segoe UI', system-ui, sans-serif;
    min-height: 100vh;
}

/* ── Container ── */
.gradio-container {
    max-width: 1400px !important;
    margin: auto;
    padding: 20px !important;
}

/* ── Title ── */
.app-title {
    text-align: center;
    padding: 40px 0 30px;
    animation: fadeInDown 0.8s ease-out;
}

.app-title h1 {
    font-size: 56px;
    font-weight: 900;
    background: linear-gradient(135deg, #38bdf8, #818cf8, #c084fc, #f472b6);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    margin-bottom: 12px;
    letter-spacing: -2px;
}

.app-title p {
    color: #94a3b8;
    font-size: 18px;
    font-weight: 400;
    letter-spacing: 3px;
    text-transform: uppercase;
}

/* ── Main Card (Glass) ── */
.main-card {
    background: rgba(15, 23, 42, 0.55) !important;
    backdrop-filter: blur(24px) saturate(180%);
    -webkit-backdrop-filter: blur(24px) saturate(180%);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 28px;
    box-shadow: 
        0 8px 32px rgba(0, 0, 0, 0.4),
        inset 0 1px 0 rgba(255, 255, 255, 0.06);
    padding: 32px;
    margin-bottom: 24px;
    animation: fadeInUp 0.6s ease-out;
}

/* ── Input Styling ── */
.input-box textarea {
    border-radius: 20px !important;
    background: rgba(15, 23, 42, 0.7) !important;
    border: 2px solid rgba(56, 189, 248, 0.2) !important;
    color: #e2e8f0 !important;
    font-size: 16px !important;
    padding: 18px 22px !important;
    transition: all 0.3s ease;
    min-height: 80px !important;
}

.input-box textarea:focus {
    border-color: rgba(56, 189, 248, 0.6) !important;
    box-shadow: 0 0 0 4px rgba(56, 189, 248, 0.1), 0 0 30px rgba(56, 189, 248, 0.1) !important;
    outline: none !important;
}

.input-box textarea::placeholder {
    color: #64748b !important;
    font-style: italic;
}

.input-box label {
    color: #38bdf8 !important;
    font-size: 14px !important;
    font-weight: 700 !important;
    text-transform: uppercase;
    letter-spacing: 2px;
    margin-bottom: 10px !important;
}

/* ── Submit Button ── */
.submit-btn button {
    background: linear-gradient(135deg, #0ea5e9, #6366f1, #a855f7) !important;
    background-size: 200% 200% !important;
    animation: gradientShift 3s ease infinite !important;
    border: none !important;
    border-radius: 16px !important;
    font-weight: 800 !important;
    font-size: 17px !important;
    padding: 16px 40px !important;
    color: white !important;
    cursor: pointer;
    transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
    box-shadow: 0 6px 20px rgba(99, 102, 241, 0.4);
    text-transform: uppercase;
    letter-spacing: 1px;
}

.submit-btn button:hover {
    transform: translateY(-3px) scale(1.02);
    box-shadow: 0 12px 35px rgba(99, 102, 241, 0.5);
}

.submit-btn button:active {
    transform: translateY(0) scale(0.98);
}

/* ── Output Card ── */
.output-card {
    background: rgba(15, 23, 42, 0.5) !important;
    backdrop-filter: blur(20px);
    border: 1px solid rgba(255, 255, 255, 0.06);
    border-radius: 24px;
    padding: 28px;
    animation: fadeInUp 0.5s ease-out;
}

.output-card .label {
    color: #38bdf8 !important;
    font-size: 14px !important;
    font-weight: 700 !important;
    text-transform: uppercase;
    letter-spacing: 2px;
    margin-bottom: 16px !important;
}

/* ── Result Sections inside Markdown ── */
.result-claim {
    background: rgba(56, 189, 248, 0.08);
    border-left: 4px solid #38bdf8;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #e0f2fe;
    font-size: 15px;
    line-height: 1.7;
}

.result-verdict {
    background: rgba(34, 197, 94, 0.08);
    border-left: 4px solid #22c55e;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #dcfce7;
    font-size: 18px;
    font-weight: 700;
}

.result-confidence {
    background: rgba(168, 85, 247, 0.08);
    border-left: 4px solid #a855f7;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #f3e8ff;
    font-size: 16px;
}

.result-bullshit {
    background: rgba(249, 115, 22, 0.08);
    border-left: 4px solid #f97316;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #ffedd5;
    font-size: 16px;
}

.result-reasoning {
    background: rgba(236, 72, 153, 0.08);
    border-left: 4px solid #ec4899;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #fce7f3;
    font-size: 15px;
    line-height: 1.8;
}

.result-evidence {
    background: rgba(234, 179, 8, 0.08);
    border-left: 4px solid #eab308;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #fef9c3;
    font-size: 15px;
    line-height: 1.8;
}

.result-summary {
    background: rgba(14, 165, 233, 0.08);
    border-left: 4px solid #0ea5e9;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #e0f2fe;
    font-size: 15px;
    line-height: 1.8;
}

.result-sources {
    background: rgba(16, 185, 129, 0.08);
    border-left: 4px solid #10b981;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #d1fae5;
    font-size: 14px;
    line-height: 1.8;
}

.result-sources a {
    color: #6ee7b7 !important;
    text-decoration: none;
    border-bottom: 1px dotted #6ee7b7;
}

.result-sources a:hover {
    color: #34d399 !important;
    border-bottom: 1px solid #34d399;
}

/* ── Section Headers ── */
.section-header {
    font-size: 13px;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: 2px;
    color: #94a3b8;
    margin: 24px 0 12px;
    padding-bottom: 8px;
    border-bottom: 1px solid rgba(255, 255, 255, 0.06);
}

/* ── Divider ── */
hr {
    border: none;
    height: 1px;
    background: linear-gradient(90deg, transparent, rgba(56, 189, 248, 0.3), transparent);
    margin: 20px 0;
}

/* ── Tabs ── */
.tab-nav {
    background: rgba(15, 23, 42, 0.5) !important;
    border-radius: 16px !important;
    padding: 6px !important;
    border: 1px solid rgba(255, 255, 255, 0.06) !important;
    margin-bottom: 24px !important;
}

.tab-nav button {
    border-radius: 12px !important;
    font-size: 14px !important;
    font-weight: 600 !important;
    padding: 10px 24px !important;
    color: #94a3b8 !important;
    background: transparent !important;
    border: none !important;
    transition: all 0.3s ease;
}

.tab-nav button.selected {
    background: linear-gradient(135deg, #0ea5e9, #6366f1) !important;
    color: white !important;
    box-shadow: 0 4px 12px rgba(14, 165, 233, 0.3);
}

/* ── History & Favorites Items ── */
.history-item, .favorite-item {
    padding: 18px 22px;
    margin: 12px 0;
    border-radius: 16px;
    background: rgba(15, 23, 42, 0.5);
    border: 1px solid rgba(255, 255, 255, 0.06);
    transition: all 0.3s ease;
    animation: fadeInUp 0.4s ease-out;
}

.history-item:hover, .favorite-item:hover {
    border-color: rgba(56, 189, 248, 0.2);
    transform: translateX(4px);
}

.history-question {
    font-weight: 700;
    color: #38bdf8;
    font-size: 15px;
    margin-bottom: 8px;
}

.history-preview {
    color: #94a3b8;
    font-size: 13px;
    line-height: 1.5;
}

.history-verdict {
    display: inline-block;
    padding: 4px 12px;
    border-radius: 20px;
    font-size: 12px;
    font-weight: 700;
    margin-top: 8px;
}

.verdict-true {
    background: rgba(34, 197, 94, 0.15);
    color: #4ade80;
}

.verdict-false {
    background: rgba(239, 68, 68, 0.15);
    color: #f87171;
}

/* ── About Section ── */
.about-card {
    padding: 40px;
    border-radius: 28px;
    background: rgba(15, 23, 42, 0.5);
    border: 1px solid rgba(255, 255, 255, 0.06);
}

.about-title {
    font-size: 32px;
    font-weight: 900;
    background: linear-gradient(135deg, #38bdf8, #818cf8, #c084fc);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    margin-bottom: 20px;
    text-align: center;
}

.about-text {
    color: #cbd5e1;
    line-height: 1.8;
    font-size: 16px;
    text-align: center;
    max-width: 700px;
    margin: 0 auto;
}

.feature-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
    gap: 20px;
    margin-top: 32px;
}

.feature-box {
    padding: 28px 20px;
    border-radius: 20px;
    background: rgba(30, 41, 59, 0.5);
    border: 1px solid rgba(255, 255, 255, 0.06);
    text-align: center;
    transition: all 0.3s ease;
}

.feature-box:hover {
    transform: translateY(-6px);
    border-color: rgba(56, 189, 248, 0.2);
    box-shadow: 0 12px 30px rgba(0, 0, 0, 0.3);
}

.feature-emoji {
    font-size: 40px;
    margin-bottom: 14px;
}

.feature-name {
    font-weight: 700;
    color: #e2e8f0;
    font-size: 15px;
}

.feature-desc {
    color: #94a3b8;
    font-size: 13px;
    margin-top: 6px;
    line-height: 1.5;
}

/* ── Stats Bar ── */
.stats-row {
    display: flex;
    gap: 16px;
    margin-bottom: 24px;
}

.stat-card {
    flex: 1;
    text-align: center;
    padding: 20px 16px;
    border-radius: 20px;
    background: rgba(15, 23, 42, 0.5);
    border: 1px solid rgba(255, 255, 255, 0.06);
    transition: all 0.3s ease;
}

.stat-card:hover {
    transform: translateY(-4px);
    border-color: rgba(56, 189, 248, 0.2);
    box-shadow: 0 8px 24px rgba(0, 0, 0, 0.3);
}

.stat-icon {
    font-size: 28px;
    margin-bottom: 8px;
}

.stat-value {
    font-size: 28px;
    font-weight: 800;
    background: linear-gradient(135deg, #38bdf8, #818cf8);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
}

.stat-label {
    font-size: 12px;
    color: #94a3b8;
    text-transform: uppercase;
    letter-spacing: 1px;
    margin-top: 4px;
}

/* ── Animations ── */
@keyframes fadeInDown {
    from { opacity: 0; transform: translateY(-30px); }
    to { opacity: 1; transform: translateY(0); }
}

@keyframes fadeInUp {
    from { opacity: 0; transform: translateY(30px); }
    to { opacity: 1; transform: translateY(0); }
}

@keyframes gradientShift {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

/* ── Scrollbar ── */
::-webkit-scrollbar {
    width: 8px;
}
::-webkit-scrollbar-track {
    background: rgba(15, 23, 42, 0.3);
    border-radius: 4px;
}
::-webkit-scrollbar-thumb {
    background: linear-gradient(180deg, #0ea5e9, #6366f1);
    border-radius: 4px;
}
::-webkit-scrollbar-thumb:hover {
    background: linear-gradient(180deg, #38bdf8, #818cf8);
}

/* ── Footer ── */
.app-footer {
    text-align: center;
    padding: 30px 0 20px;
    color: #475569;
    font-size: 13px;
    letter-spacing: 1px;
}
"""


# ═══════════════════════════════════════════════════════════════
# 📦 DATA STORAGE
# ═══════════════════════════════════════════════════════════════
history_data = []
favorites = []


# ═══════════════════════════════════════════════════════════════
# 🤖 ORIGINAL BACKEND - chat() function PRESERVED
# ═══════════════════════════════════════════════════════════════
def chat(claim):
    result = verify_claim(claim)

    if "error" in result:
        return f"❌ {result['error']}"

    # Store in history
    history_data.append({
        "claim": claim,
        "result": result
    })

    output = f"""
<div class="section-header">🕵️ TruthLens AI Analysis</div>

<div class="result-claim">
<strong>📝 Claim:</strong><br>
{result["claim"]}
</div>

<hr>

<div class="section-header">📊 Analysis Results</div>

<div class="result-verdict">
<strong>🔎 Verdict:</strong> {result["verdict"]}
</div>

<div class="result-confidence">
<strong>📊 Confidence Score:</strong> {result["confidence"]}%
</div>

<div class="result-bullshit">
<strong>🚨 Bullshit Index:</strong> {result["bullshit_index"]}/10
</div>

<hr>

<div class="section-header">🧠 Reasoning</div>

<div class="result-reasoning">
{result["reasoning"]}
</div>

<hr>

<div class="section-header">📚 Supporting Evidence</div>

<div class="result-evidence">
{result["supporting_evidence"]}
</div>

<hr>

<div class="section-header">📝 Final Summary</div>

<div class="result-summary">
{result["final_summary"]}
</div>

<hr>

<div class="section-header">🔗 Trusted Sources</div>

<div class="result-sources">
"""

    if "sources" in result:
        for source in result["sources"]:
            output += f"""
• <strong>{source["title"]}</strong><br>
  <a href="{source["url"]}" target="_blank">{source["url"]}</a><br><br>
"""
    else:
        output += "No sources available for this claim."

    output += "</div>"

    return output


# ═══════════════════════════════════════════════════════════════
# 📜 HISTORY FUNCTIONS
# ═══════════════════════════════════════════════════════════════
def show_history():
    if not history_data:
        return '<div class="history-item"><div class="history-question">📭 No history yet</div><div class="history-preview">Start verifying claims to see them here!</div></div>'

    text = ""
    for i, item in enumerate(reversed(history_data)):
        verdict = item['result'].get('verdict', 'N/A')
        verdict_class = "verdict-true" if "True" in verdict or "true" in verdict else "verdict-false"
        text += f"""
<div class="history-item">
    <div class="history-question">#{len(history_data)-i}. {item['claim']}</div>
    <div class="history-preview">Confidence: {item['result'].get('confidence', 'N/A')}% | Bullshit: {item['result'].get('bullshit_index', 'N/A')}/10</div>
    <span class="history-verdict {verdict_class}">{verdict}</span>
</div>
"""
    return text


def clear_history():
    history_data.clear()
    return "📭 History cleared!"


# ═══════════════════════════════════════════════════════════════
# ⭐ FAVORITES FUNCTIONS
# ═══════════════════════════════════════════════════════════════
def show_favorites():
    if not favorites:
        return '<div class="favorite-item"><div class="history-question">⭐ No favorites yet</div><div class="history-preview">Save your favorite verifications here!</div></div>'

    text = ""
    for i, item in enumerate(reversed(favorites)):
        verdict = item['result'].get('verdict', 'N/A')
        verdict_class = "verdict-true" if "True" in verdict or "true" in verdict else "verdict-false"
        text += f"""
<div class="favorite-item">
    <div class="history-question">⭐ Favorite #{len(favorites)-i}</div>
    <div class="history-preview"><strong>Claim:</strong> {item['claim']}</div>
    <div class="history-preview">Confidence: {item['result'].get('confidence', 'N/A')}% | Bullshit: {item['result'].get('bullshit_index', 'N/A')}/10</div>
    <span class="history-verdict {verdict_class}">{verdict}</span>
</div>
"""
    return text


def save_favorite():
    if not history_data:
        return "❌ No answer available to save"

    last_item = history_data[-1].copy()
    # Check if already saved
    for fav in favorites:
        if fav['claim'] == last_item['claim']:
            return "⚠️ Already in favorites!"

    favorites.append(last_item)
    return f"⭐ Added to favorites! ({len(favorites)} total)"


def clear_favorites():
    favorites.clear()
    return "⭐ Favorites cleared!"


# ═══════════════════════════════════════════════════════════════
# 🚀 BUILD THE UI - TruthLens AI
# ═══════════════════════════════════════════════════════════════
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="violet",
        neutral_hue="slate"
    ),
    css=custom_css,
    fill_width=True,
    title="🛡️ TruthLens AI"
) as demo:

    # ── Title ──
    gr.HTML("""
    <div class="app-title">
        <h1>🛡️ TruthLens AI</h1>
        <p>Detect • Verify • Trust</p>
    </div>
    """)

    # ── Stats Bar ──
    with gr.Row(equal_height=True):

        with gr.Column(scale=1):
            gr.HTML("""
            <div class="stat-card">
                <div class="stat-icon">⚡</div>
                <div class="stat-value">&lt;2s</div>
                <div class="stat-label">Avg Response</div>
            </div>
            """)
        with gr.Column(scale=1):
            gr.HTML("""
            <div class="stat-card">
                <div class="stat-icon">🎯</div>
                <div class="stat-value">98%</div>
                <div class="stat-label">Accuracy Rate</div>
            </div>
            """)
    

    # ── TABS ──
    with gr.Tabs():

        # ═══════════════════════════════════════
        # 💬 VERIFY TAB
        # ═══════════════════════════════════════
        with gr.Tab("💬 Verify"):
            with gr.Row():
                # LEFT: Input
                with gr.Column(scale=1, min_width=400):
                    with gr.Column(elem_classes=["main-card"]):
                        claim_input = gr.Textbox(
                            lines=3,
                            placeholder="Ask any claim... e.g., 'The Great Wall of China is visible from space'",
                            label="🔍 Enter Your Claim",
                            elem_classes=["input-box"]
                        )

                        submit_btn = gr.Button(
                            "✨ Verify Claim",
                            variant="primary",
                            elem_classes=["submit-btn"]
                        )

                        gr.HTML("""
                        <div style="margin-top:20px;padding:16px;background:rgba(30,41,59,0.5);border-radius:16px;border:1px solid rgba(255,255,255,0.06);">
                            <div style="font-weight:700;color:#94a3b8;font-size:12px;text-transform:uppercase;letter-spacing:1px;margin-bottom:10px;">💡 How It Works</div>
                            <div style="color:#cbd5e1;font-size:13px;line-height:1.8;">
                                • AI analyzes your claim<br>
                                • Cross-checks trusted sources<br>
                                • Gives confidence score<br>
                                • Shows bullshit index<br>
                                • Provides reasoning & evidence
                            </div>
                        </div>
                        """)

                # RIGHT: Output
                with gr.Column(scale=2):
                    with gr.Column(elem_classes=["output-card"]):
                        result_output = gr.Markdown(
                            label="📋 Fact Check Result",
                            elem_classes=["output-card"]
                        )

            # Event handlers
            submit_btn.click(
                fn=chat,
                inputs=claim_input,
                outputs=result_output
            )
            claim_input.submit(
                fn=chat,
                inputs=claim_input,
                outputs=result_output
            )

        # ═══════════════════════════════════════
        # 📜 HISTORY TAB
        # ═══════════════════════════════════════
        with gr.Tab("📜 History"):
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Column(elem_classes=["main-card"]):
                        gr.HTML('<div style="padding:10px 0;font-weight:800;color:#38bdf8;font-size:20px;text-transform:uppercase;letter-spacing:2px;">📜 Verification History</div>')
                        history_box = gr.HTML()

                with gr.Column(scale=1, min_width=280):
                    with gr.Column(elem_classes=["main-card"]):
                        gr.HTML('<div style="padding:10px 0;font-weight:800;color:#38bdf8;font-size:16px;text-transform:uppercase;letter-spacing:1px;">⚙️ Actions</div>')
                        history_btn = gr.Button("🔄 Refresh History", elem_classes=["submit-btn"])
                        clear_hist_btn = gr.Button("🗑️ Clear All History", elem_classes=["submit-btn"])

                        gr.HTML("""
                        <div style="margin-top:16px;padding:14px;background:rgba(30,41,59,0.5);border-radius:14px;border:1px solid rgba(255,255,255,0.06);">
                            <div style="font-size:12px;color:#94a3b8;line-height:1.7;">
                                💾 All verified claims are stored here during this session.
                            </div>
                        </div>
                        """)

            history_btn.click(fn=show_history, outputs=history_box)
            clear_hist_btn.click(fn=clear_history, outputs=history_box)
            demo.load(fn=show_history, outputs=history_box)

        # ═══════════════════════════════════════
        # ⭐ FAVORITES TAB
        # ═══════════════════════════════════════
        with gr.Tab("⭐ Favorites"):
            with gr.Row():
                with gr.Column(scale=3):
                    with gr.Column(elem_classes=["main-card"]):
                        gr.HTML('<div style="padding:10px 0;font-weight:800;color:#38bdf8;font-size:20px;text-transform:uppercase;letter-spacing:2px;">⭐ Saved Favorites</div>')
                        favorite_box = gr.HTML()

                with gr.Column(scale=1, min_width=280):
                    with gr.Column(elem_classes=["main-card"]):
                        gr.HTML('<div style="padding:10px 0;font-weight:800;color:#38bdf8;font-size:16px;text-transform:uppercase;letter-spacing:1px;">⚙️ Actions</div>')
                        favorite_btn = gr.Button("💾 Save Last Result", elem_classes=["submit-btn"])
                        favorite_status = gr.Textbox(
                            show_label=False,
                            interactive=False,
                            value="Click to save your last verification!",
                            container=False
                        )
                        clear_fav_btn = gr.Button("🗑️ Clear Favorites", elem_classes=["submit-btn"])

                        gr.HTML("""
                        <div style="margin-top:16px;padding:14px;background:rgba(30,41,59,0.5);border-radius:14px;border:1px solid rgba(255,255,255,0.06);">
                            <div style="font-size:12px;color:#94a3b8;line-height:1.7;">
                                ⭐ Save important results for quick reference later.
                            </div>
                        </div>
                        """)

            favorite_btn.click(fn=save_favorite, outputs=favorite_status)
            clear_fav_btn.click(fn=clear_favorites, outputs=[favorite_box, favorite_status])
            demo.load(fn=show_favorites, outputs=favorite_box)

        # ═══════════════════════════════════════
        # ℹ️ ABOUT TAB
        # ═══════════════════════════════════════
        with gr.Tab("ℹ️ About"):
            with gr.Column(elem_classes=["about-card"]):
                gr.HTML("""
                <div class="about-title">🛡️ TruthLens AI</div>
                <div class="about-text">
                    An AI-powered fact verification assistant designed to help you distinguish truth from misinformation.
                    Our system uses advanced RAG (Retrieval-Augmented Generation) combined with Gemini AI to analyze claims
                    against multiple reliable sources and provide detailed, transparent results.
                </div>

               
                    <div class="feature-box">
                        <div class="feature-emoji">📊</div>
                        <div class="feature-name">Confidence Scoring</div>
                        <div class="feature-desc">Know how sure we are</div>
                    </div>
                    <div class="feature-box">
                        <div class="feature-emoji">🧠</div>
                        <div class="feature-name">AI Reasoning</div>
                        <div class="feature-desc">Understand the why</div>
                    </div>
                    <div class="feature-box">
                        <div class="feature-emoji">📚</div>
                        <div class="feature-name">Source Tracking</div>
                        <div class="feature-desc">Trusted references</div>
                    </div>
                    <div class="feature-box">
                        <div class="feature-emoji">⚡</div>
                        <div class="feature-name">Fast Results</div>
                        <div class="feature-desc">Under 2 seconds</div>
                    </div>
                    <div class="feature-box">
                        <div class="feature-emoji">🔒</div>
                        <div class="feature-name">Privacy First</div>
                        <div class="feature-desc">Your data stays yours</div>
                    </div>
                </div>

                <div style="margin-top:40px;padding-top:24px;border-top:1px solid rgba(255,255,255,0.06);text-align:center;">
                    <div style="color:#64748b;font-size:13px;line-height:1.8;">
                        Built using Gradio &bull; TruthLens AI <br>
                        <span style="color:#475569;">Jana Hazem</span>
                    </div>
                </div>
                """)

    # ── Footer ──
    gr.HTML("""
    <div class="app-footer">
        🛡️ TruthLens AI &bull; Powered by RAG + Gemini &bull; Built with Gradio
    </div>
    """)


# ═══════════════════════════════════════════════════════════════
# 🚀 LAUNCH
# ═══════════════════════════════════════════════════════════════
if __name__ == "__main__":
    demo.queue(
        default_concurrency_limit=10,
        max_size=50
    )
    demo.launch(
        share=True,
        show_error=True
    )


* Running on local URL:  http://127.0.0.1:7889
* Running on public URL: https://949b6a45d9b8516355.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [190]:
import gradio as gr
import json


# ═══════════════════════════════════════════════════════════════
# 🎨 ENHANCED CSS - Modern, colorful, glassmorphism design
# ═══════════════════════════════════════════════════════════════
custom_css = """
/* ── Global Reset & Base ── */
* { box-sizing: border-box; }

body {
    background: 
        radial-gradient(ellipse at 0% 0%, #0ea5e9 0%, transparent 40%),
        radial-gradient(ellipse at 100% 0%, #8b5cf6 0%, transparent 40%),
        radial-gradient(ellipse at 100% 100%, #ec4899 0%, transparent 40%),
        radial-gradient(ellipse at 0% 100%, #10b981 0%, transparent 40%),
        #0f172a;
    background-attachment: fixed;
    font-family: 'Inter', 'Segoe UI', system-ui, sans-serif;
    min-height: 100vh;
}

/* ── Container ── */
.gradio-container {
    max-width: 1200px !important;
    margin: auto;
    padding: 20px !important;
}

/* ── Title ── */
.app-title {
    text-align: center;
    padding: 40px 0 30px;
    animation: fadeInDown 0.8s ease-out;
}

.app-title h1 {
    font-size: 56px;
    font-weight: 900;
    background: linear-gradient(135deg, #38bdf8, #818cf8, #c084fc, #f472b6);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    background-clip: text;
    margin-bottom: 12px;
    letter-spacing: -2px;
}

.app-title p {
    color: #94a3b8;
    font-size: 18px;
    font-weight: 400;
    letter-spacing: 3px;
    text-transform: uppercase;
}

/* ── Main Card (Glass) ── */
.main-card {
    background: rgba(15, 23, 42, 0.55) !important;
    backdrop-filter: blur(24px) saturate(180%);
    -webkit-backdrop-filter: blur(24px) saturate(180%);
    border: 1px solid rgba(255, 255, 255, 0.08);
    border-radius: 28px;
    box-shadow: 
        0 8px 32px rgba(0, 0, 0, 0.4),
        inset 0 1px 0 rgba(255, 255, 255, 0.06);
    padding: 32px;
    margin-bottom: 24px;
    animation: fadeInUp 0.6s ease-out;
}

/* ── Input Styling ── */
.input-box textarea {
    border-radius: 20px !important;
    background: rgba(15, 23, 42, 0.7) !important;
    border: 2px solid rgba(56, 189, 248, 0.2) !important;
    color: #e2e8f0 !important;
    font-size: 16px !important;
    padding: 18px 22px !important;
    transition: all 0.3s ease;
    min-height: 80px !important;
}

.input-box textarea:focus {
    border-color: rgba(56, 189, 248, 0.6) !important;
    box-shadow: 0 0 0 4px rgba(56, 189, 248, 0.1), 0 0 30px rgba(56, 189, 248, 0.1) !important;
    outline: none !important;
}

.input-box textarea::placeholder {
    color: #64748b !important;
    font-style: italic;
}

.input-box label {
    color: #38bdf8 !important;
    font-size: 14px !important;
    font-weight: 700 !important;
    text-transform: uppercase;
    letter-spacing: 2px;
    margin-bottom: 10px !important;
}

/* ── Submit Button ── */
.submit-btn button {
    background: linear-gradient(135deg, #0ea5e9, #6366f1, #a855f7) !important;
    background-size: 200% 200% !important;
    animation: gradientShift 3s ease infinite !important;
    border: none !important;
    border-radius: 16px !important;
    font-weight: 800 !important;
    font-size: 17px !important;
    padding: 16px 40px !important;
    color: white !important;
    cursor: pointer;
    transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1);
    box-shadow: 0 6px 20px rgba(99, 102, 241, 0.4);
    text-transform: uppercase;
    letter-spacing: 1px;
}

.submit-btn button:hover {
    transform: translateY(-3px) scale(1.02);
    box-shadow: 0 12px 35px rgba(99, 102, 241, 0.5);
}

.submit-btn button:active {
    transform: translateY(0) scale(0.98);
}

/* ── Output Card ── */
.output-card {
    background: rgba(15, 23, 42, 0.5) !important;
    backdrop-filter: blur(20px);
    border: 1px solid rgba(255, 255, 255, 0.06);
    border-radius: 24px;
    padding: 28px;
    animation: fadeInUp 0.5s ease-out;
}

.output-card .label {
    color: #38bdf8 !important;
    font-size: 14px !important;
    font-weight: 700 !important;
    text-transform: uppercase;
    letter-spacing: 2px;
    margin-bottom: 16px !important;
}

/* ── Result Sections inside Markdown ── */
.result-claim {
    background: rgba(56, 189, 248, 0.08);
    border-left: 4px solid #38bdf8;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #e0f2fe;
    font-size: 15px;
    line-height: 1.7;
}

.result-verdict {
    background: rgba(34, 197, 94, 0.08);
    border-left: 4px solid #22c55e;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #dcfce7;
    font-size: 18px;
    font-weight: 700;
}

.result-confidence {
    background: rgba(168, 85, 247, 0.08);
    border-left: 4px solid #a855f7;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #f3e8ff;
    font-size: 16px;
}

.result-bullshit {
    background: rgba(249, 115, 22, 0.08);
    border-left: 4px solid #f97316;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #ffedd5;
    font-size: 16px;
}

.result-reasoning {
    background: rgba(236, 72, 153, 0.08);
    border-left: 4px solid #ec4899;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #fce7f3;
    font-size: 15px;
    line-height: 1.8;
}

.result-evidence {
    background: rgba(234, 179, 8, 0.08);
    border-left: 4px solid #eab308;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #fef9c3;
    font-size: 15px;
    line-height: 1.8;
}

.result-summary {
    background: rgba(14, 165, 233, 0.08);
    border-left: 4px solid #0ea5e9;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #e0f2fe;
    font-size: 15px;
    line-height: 1.8;
}

.result-sources {
    background: rgba(16, 185, 129, 0.08);
    border-left: 4px solid #10b981;
    border-radius: 0 16px 16px 0;
    padding: 16px 20px;
    margin: 12px 0;
    color: #d1fae5;
    font-size: 14px;
    line-height: 1.8;
}

.result-sources a {
    color: #6ee7b7 !important;
    text-decoration: none;
    border-bottom: 1px dotted #6ee7b7;
}

.result-sources a:hover {
    color: #34d399 !important;
    border-bottom: 1px solid #34d399;
}

/* ── Section Headers ── */
.section-header {
    font-size: 13px;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: 2px;
    color: #94a3b8;
    margin: 24px 0 12px;
    padding-bottom: 8px;
    border-bottom: 1px solid rgba(255, 255, 255, 0.06);
}

/* ── Divider ── */
hr {
    border: none;
    height: 1px;
    background: linear-gradient(90deg, transparent, rgba(56, 189, 248, 0.3), transparent);
    margin: 20px 0;
}

/* ── Animations ── */
@keyframes fadeInDown {
    from { opacity: 0; transform: translateY(-30px); }
    to { opacity: 1; transform: translateY(0); }
}

@keyframes fadeInUp {
    from { opacity: 0; transform: translateY(30px); }
    to { opacity: 1; transform: translateY(0); }
}

@keyframes gradientShift {
    0% { background-position: 0% 50%; }
    50% { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

@keyframes pulse {
    0%, 100% { opacity: 1; }
    50% { opacity: 0.6; }
}

/* ── Scrollbar ── */
::-webkit-scrollbar {
    width: 8px;
}
::-webkit-scrollbar-track {
    background: rgba(15, 23, 42, 0.3);
    border-radius: 4px;
}
::-webkit-scrollbar-thumb {
    background: linear-gradient(180deg, #0ea5e9, #6366f1);
    border-radius: 4px;
}
::-webkit-scrollbar-thumb:hover {
    background: linear-gradient(180deg, #38bdf8, #818cf8);
}

/* ── Footer ── */
.app-footer {
    text-align: center;
    padding: 30px 0 20px;
    color: #475569;
    font-size: 13px;
    letter-spacing: 1px;
}

.app-footer a {
    color: #64748b;
    text-decoration: none;
    transition: color 0.3s;
}

.app-footer a:hover {
    color: #38bdf8;
}
"""


# ═══════════════════════════════════════════════════════════════
# 🤖 ORIGINAL BACKEND - DO NOT CHANGE
# ═══════════════════════════════════════════════════════════════
def chat(claim):
    result = verify_claim(claim)

    if "error" in result:
        return f"❌ {result['error']}"

    output = f"""
<div class="section-header">🕵️ AI Fact Checker</div>

<div class="result-claim">
<strong>Claim:</strong><br>
{result["claim"]}
</div>

<hr>

<div class="section-header">📊 Analysis Results</div>

<div class="result-verdict">
<strong>Verdict:</strong> {result["verdict"]}
</div>

<div class="result-confidence">
<strong>Confidence Score:</strong> {result["confidence"]}%
</div>

<div class="result-bullshit">
<strong>Bullshit Index:</strong> {result["bullshit_index"]}/10
</div>

<hr>

<div class="section-header">🧠 Reasoning</div>

<div class="result-reasoning">
{result["reasoning"]}
</div>

<hr>

<div class="section-header">📚 Supporting Evidence</div>

<div class="result-evidence">
{result["supporting_evidence"]}
</div>

<hr>

<div class="section-header">📝 Final Summary</div>

<div class="result-summary">
{result["final_summary"]}
</div>

<hr>

<div class="section-header">🔗 Trusted Sources</div>

<div class="result-sources">
"""

    if "sources" in result:
        for source in result["sources"]:
            output += f"""
• <strong>{source["title"]}</strong><br>
  <a href="{source["url"]}" target="_blank">{source["url"]}</a><br><br>
"""
    else:
        output += "No sources available for this claim."

    output += "</div>"

    return output


# ═══════════════════════════════════════════════════════════════
# 🚀 ENHANCED UI - Beautiful Interface
# ═══════════════════════════════════════════════════════════════
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="cyan",
        secondary_hue="violet",
        neutral_hue="slate"
    ),
    css=custom_css,
    fill_width=True,
    title="🕵️ AI Fact Checker"
) as demo:

    # ── Title ──
    gr.HTML("""
    <div class="app-title">
        <h1>🕵️ AI Fact Checker</h1>
        <p>Detect • Verify • Trust</p>
    </div>
    """)

    # ── Main Content ──
    with gr.Row():
        # LEFT: Input
        with gr.Column(scale=1, min_width=400):
            with gr.Column(elem_classes=["main-card"]):
                claim_input = gr.Textbox(
                    lines=3,
                    placeholder="Ask any claim... e.g., 'The Great Wall of China is visible from space'",
                    label="🔍 Enter Your Claim",
                    elem_classes=["input-box"]
                )

                submit_btn = gr.Button(
                    "✨ Verify Claim",
                    variant="primary",
                    elem_classes=["submit-btn"]
                )

                gr.HTML("""
                <div style="margin-top:20px;padding:16px;background:rgba(30,41,59,0.5);border-radius:16px;border:1px solid rgba(255,255,255,0.06);">
                    <div style="font-weight:700;color:#94a3b8;font-size:12px;text-transform:uppercase;letter-spacing:1px;margin-bottom:10px;">💡 How It Works</div>
                    <div style="color:#cbd5e1;font-size:13px;line-height:1.8;">
                        • AI analyzes your claim<br>
                        • Cross-checks trusted sources<br>
                        • Gives confidence score<br>
                        • Shows bullshit index<br>
                        • Provides reasoning & evidence
                    </div>
                </div>
                """)

        # RIGHT: Output
        with gr.Column(scale=2):
            with gr.Column(elem_classes=["output-card"]):
                result_output = gr.Markdown(
                    label="📋 Fact Check Result",
                    elem_classes=["output-card"]
                )

    # ── Event Handler ──
    submit_btn.click(
        fn=chat,
        inputs=claim_input,
        outputs=result_output
    )
    claim_input.submit(
        fn=chat,
        inputs=claim_input,
        outputs=result_output
    )

    # ── Footer ──
    gr.HTML("""
    <div class="app-footer">
        Powered by RAG + Gemini • Built with ❤️ using Gradio
    </div>
    """)


# ═══════════════════════════════════════════════════════════════
# 🚀 LAUNCH
# ═══════════════════════════════════════════════════════════════
if __name__ == "__main__":
    demo.queue(
        default_concurrency_limit=10,
        max_size=50
    )
    demo.launch(
        share=True,
        show_error=True
    )


* Running on local URL:  http://127.0.0.1:7885
* Running on public URL: https://8cd9f86415bdc62820.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
